# 03 — EfficientNet-B0 Product Category Classifier
## Brand Vision — Two-Phase Fine-Tuning

- **Phase 1 (5 epochs):** Train only the classifier head (backbone frozen)
- **Phase 2 (30 epochs):** Unfreeze full network, fine-tune end-to-end
- **Batch size:** 32 locally (RTX 4050), 64 on Kaggle P100
- **Target:** val accuracy ≥ 0.65 (ideally ≥ 0.80)

In [1]:
# Cell 1 — Imports
import os, json
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.amp import GradScaler, autocast

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ON_KAGGLE = os.path.exists("/kaggle")
BATCH_SIZE = 64 if ON_KAGGLE else 32   # 32 is safe for 6GB VRAM with AMP

print(f"Device: {device} | Batch: {BATCH_SIZE} | Platform: {'Kaggle' if ON_KAGGLE else 'Local'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda | Batch: 32 | Platform: Local
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
# Cell 2 — Config — UPDATE PATHS IF NEEDED
#   Local:  relative path from training/ directory
#   Kaggle: "/kaggle/working/product_category_split"
DATA_DIR = Path("datasets/product_category_split")
SAVE_DIR = Path("runs/category")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Cell 3 — Transforms
train_tfm = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(str(DATA_DIR / "train"), train_tfm)
val_ds   = datasets.ImageFolder(str(DATA_DIR / "val"),   val_tfm)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

NUM_CLASSES = len(train_ds.classes)
print(f"Classes ({NUM_CLASSES}): {train_ds.classes}")
print(f"Train: {len(train_ds)} images | Val: {len(val_ds)} images")

Classes (12): ['backpack', 'belt', 'cap', 'dress', 'handbag', 'jacket', 'jeans', 'shoes', 'sunglasses', 't-shirt', 'wallet', 'watch']
Train: 9039 images | Val: 2583 images


In [4]:
# Cell 4 — Build model
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze backbone — only train head in Phase 1
for p in model.features.parameters():
    p.requires_grad = False

in_feats = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_feats, NUM_CLASSES),
)
model = model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params (Phase 1): {trainable:,}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\mailm/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|████████████████████████████████████████████████████████████████████████████| 20.5M/20.5M [00:04<00:00, 4.63MB/s]


Trainable params (Phase 1): 15,372


In [5]:
# Cell 5 — Training loop helper
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler    = GradScaler("cuda")
best_acc  = 0.0
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)

def run_epoch(loader, train: bool) -> tuple[float, float]:
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()

    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
                with autocast("cuda"):
                    out  = model(imgs)
                    loss = criterion(out, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                with autocast("cuda"):
                    out  = model(imgs)
                    loss = criterion(out, labels)

            total_loss += loss.item()
            correct    += out.argmax(1).eq(labels).sum().item()
            total      += labels.size(0)

    return total_loss / len(loader), correct / total

In [6]:
# Cell 6 — Phase 1: Head warmup (5 epochs)
print("── Phase 1: Training classifier head (backbone frozen) ──")
for epoch in range(5):
    tr_loss, tr_acc = run_epoch(train_dl, train=True)
    vl_loss, vl_acc = run_epoch(val_dl,   train=False)
    print(f"Ep {epoch+1:02d} | tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f}")

── Phase 1: Training classifier head (backbone frozen) ──
Ep 01 | tr_loss=1.1254 tr_acc=0.8164 | vl_loss=0.7332 vl_acc=0.9539
Ep 02 | tr_loss=0.8655 tr_acc=0.9009 | vl_loss=0.7133 vl_acc=0.9628
Ep 03 | tr_loss=0.8358 tr_acc=0.9144 | vl_loss=0.7068 vl_acc=0.9605
Ep 04 | tr_loss=0.8258 tr_acc=0.9156 | vl_loss=0.7030 vl_acc=0.9628
Ep 05 | tr_loss=0.8226 tr_acc=0.9147 | vl_loss=0.7017 vl_acc=0.9667


In [7]:
# Cell 7 — Phase 2: Fine-tune full network (30 epochs)
for p in model.features.parameters():
    p.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

print("── Phase 2: Fine-tuning full network (30 epochs) ──")
for epoch in range(30):
    tr_loss, tr_acc = run_epoch(train_dl, train=True)
    vl_loss, vl_acc = run_epoch(val_dl,   train=False)
    scheduler.step()

    print(f"Ep {epoch+1:02d} | tr_acc={tr_acc:.4f} | vl_acc={vl_acc:.4f} | lr={scheduler.get_last_lr()[0]:.6f}")

    if vl_acc > best_acc:
        best_acc = vl_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_acc": vl_acc,
            "classes": train_ds.classes,
        }, str(SAVE_DIR / "best_model.pt"))
        print(f"  ✅ Saved best model (val_acc={vl_acc:.4f})")

print(f"\nBest validation accuracy: {best_acc:.4f}")
status = "✅" if best_acc >= 0.65 else "⚠"
print(f"{status} Target: ≥0.65 | Ideal: ≥0.80")

── Phase 2: Fine-tuning full network (30 epochs) ──
Ep 01 | tr_acc=0.9521 | vl_acc=0.9795 | lr=0.000100
  ✅ Saved best model (val_acc=0.9795)
Ep 02 | tr_acc=0.9736 | vl_acc=0.9872 | lr=0.000099
  ✅ Saved best model (val_acc=0.9872)
Ep 03 | tr_acc=0.9824 | vl_acc=0.9876 | lr=0.000098
  ✅ Saved best model (val_acc=0.9876)
Ep 04 | tr_acc=0.9847 | vl_acc=0.9884 | lr=0.000096
  ✅ Saved best model (val_acc=0.9884)
Ep 05 | tr_acc=0.9868 | vl_acc=0.9872 | lr=0.000093
Ep 06 | tr_acc=0.9910 | vl_acc=0.9892 | lr=0.000090
  ✅ Saved best model (val_acc=0.9892)
Ep 07 | tr_acc=0.9908 | vl_acc=0.9884 | lr=0.000087
Ep 08 | tr_acc=0.9935 | vl_acc=0.9895 | lr=0.000083
  ✅ Saved best model (val_acc=0.9895)
Ep 09 | tr_acc=0.9951 | vl_acc=0.9899 | lr=0.000079
  ✅ Saved best model (val_acc=0.9899)
Ep 10 | tr_acc=0.9942 | vl_acc=0.9895 | lr=0.000075
Ep 11 | tr_acc=0.9949 | vl_acc=0.9899 | lr=0.000070
Ep 12 | tr_acc=0.9956 | vl_acc=0.9907 | lr=0.000065
  ✅ Saved best model (val_acc=0.9907)
Ep 13 | tr_acc=0.995

In [9]:
# Cell 8 — Export to ONNX
checkpoint = torch.load(str(SAVE_DIR / "best_model.pt"), weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval().cpu()

dummy = torch.randn(1, 3, 224, 224)
onnx_path = str(SAVE_DIR / "category_classifier.onnx")

torch.onnx.export(
    model, dummy, onnx_path,
    input_names=["image"], output_names=["logits"],
    dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)

labels_path = str(SAVE_DIR / "category_classifier_labels.json")
with open(labels_path, "w") as f:
    json.dump(train_ds.classes, f, indent=2)

print(f"ONNX saved:   {onnx_path}")
print(f"Labels saved: {labels_path}")
print(f"Classes ({NUM_CLASSES}): {train_ds.classes}")

C:\Users\mailm\AppData\Local\Temp\ipykernel_20324\1023068948.py:9: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0324 23:01:14.779000 20324 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0324 23:01:15.752000 20324 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\mailm\AppData\Local\Python\pythoncore-3.14-64\Lib\copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Projects\INT428 DL CA01\brand-vision\venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "C:\Projects\INT428 DL CA01\brand-vision\venv\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "C:\Projects\INT428 DL CA01\brand-vision\venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "C:\Projects\INT428 DL CA01\brand-vision\venv\Lib\site-packages\onnx\version_conve

[torch.onnx] Optimize the ONNX graph...
Applied 99 of general pattern rewrite rules.
[torch.onnx] Optimize the ONNX graph... ✅
ONNX saved:   runs\category\category_classifier.onnx
Labels saved: runs\category\category_classifier_labels.json
Classes (12): ['backpack', 'belt', 'cap', 'dress', 'handbag', 'jacket', 'jeans', 'shoes', 'sunglasses', 't-shirt', 'wallet', 'watch']
